# 第 24 课｜撮合机制全景

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`24_youtube-nGSSR6jEM4s/index.html`  
> **所属模块**：P1-M5 · 市场微观结构与执行

## 本节量化问题

同一批订单在价格优先、时间优先规则下如何成交？

## 准确定义与策略报告位置

更优价格优先；同价位更早进入队列的订单通常优先。撮合结果取决于订单类型、时间和剩余数量。

- **核心关系**：连续竞价通常先按价格排序，再按同价位进入时间排序；成交量=min(买方剩余量,卖方剩余量)。
- **写入报告**：成交模型、排队假设和未成交风险
- **P2 衔接**：P2 第 45–46、51、56 课：状态存储、流水线和测试。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 用价格—时间优先匹配限价单

示例只实现买卖限价单。买方按高价优先、同价早到优先；卖方相反。

In [2]:
orders = [
    {"id": "S1", "side": "sell", "price": 100.20, "qty": 300, "time": 1},
    {"id": "S2", "side": "sell", "price": 100.20, "qty": 200, "time": 2},
    {"id": "S3", "side": "sell", "price": 100.30, "qty": 400, "time": 3},
    {"id": "B1", "side": "buy", "price": 100.25, "qty": 450, "time": 4},
]

sell_book = sorted([o.copy() for o in orders if o["side"] == "sell"], key=lambda o: (o["price"], o["time"]))
buy = [o.copy() for o in orders if o["id"] == "B1"][0]
trades = []

for sell in sell_book:
    if buy["qty"] == 0 or buy["price"] < sell["price"]:
        break
    matched = min(buy["qty"], sell["qty"])
    trades.append({"buy_id": buy["id"], "sell_id": sell["id"], "price": sell["price"], "qty": matched})
    buy["qty"] -= matched
    sell["qty"] -= matched

display(pd.DataFrame(trades))
print("B1 未成交数量:", buy["qty"])
assert [trade["sell_id"] for trade in trades] == ["S1", "S2"]

,buy_id,sell_id,price,qty
0,B1,S1,100.2,300
1,B1,S2,100.2,150


B1 未成交数量: 0


## 方法边界与阶段验收

交易所细则可能包含集合竞价、涨跌停、自成交保护和特殊订单，简化引擎不能直接代表真实交易所。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
